# Chapter 05 · Notebook 05: Our own PyTorch model class

**Optional math reference:** [Mathematical view](00_math_intuition.ipynb), available for later. These programming lessons can be followed independently.

[Chapter guide](README.md) · [Previous notebook](04_python_classes.ipynb) · [Course index](../README.md)

We know how objects store their own attributes and use `self` in methods. Now define a class that stores a PyTorch layer, describes its forward calculation, and exposes the layer's parameters to SGD.

We keep the familiar model: two input features, one output, two weights and one bias. This notebook changes how the model is organized; the prediction and training calculations stay the same.

## Contents

1. [Inherit PyTorch model features](#inheritance)
2. [Store a layer and define forward](#model-class)
3. [Make familiar predictions](#predictions)
4. [Train the model with the same SGD step](#training)
5. [Inspect the trained model state](#model-state)
6. [Load the state into another model](#load-state)
7. [Save the current state to a checkpoint file](#save-file)
8. [Restore a model from the checkpoint file](#restore-file)
9. [Inspect optimizer memory for resuming training](#optimizer-state)
10. [Save a checkpoint for the momentum experiment](#training-checkpoint)
11. [Restore the model and optimizer together](#restore-training)
12. [Continue with the next momentum update](#resume-update)
13. [Compare restoring both states with restoring only the model](#resume-comparison)

Work one lesson at a time with the mentoring conversation. Start with Setup in a fresh kernel. This notebook supplies its own variables. Start Jupyter from the repository root or a chapter folder; Setup locates the chapter checkpoints without a machine-specific path.

## Setup: import PyTorch

Python supplies class syntax. PyTorch supplies `Module`, `Linear`, the tensors, loss function, and optimizer we will use.


In [ ]:
import torch
from pathlib import Path

# Find the course from either the repository root or a chapter folder.
# Start Jupyter inside this repository so a parent contains the chapter folder.
def course_checkpoint_dir():
    for directory in (Path.cwd(), *Path.cwd().parents):
        chapter = directory / "05_pytorch_model_layers"
        if chapter.is_dir():
            return chapter / "checkpoints"
    raise FileNotFoundError("Start Jupyter from inside the course repository.")

print("PyTorch version:", torch.__version__)


<a id="inheritance"></a>
## Lesson 01: inherit features from a parent class

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

**Inheritance** lets a new class use features defined by another class. In `class EmptyModel(torch.nn.Module):`, `EmptyModel` is our new class and `torch.nn.Module` is its parent (also called its base class).

`Module` provides model methods such as `parameters()`. This lets PyTorch find registered parameters belonging to the model and its layers. We do not have to write that method ourselves.

Start with an empty class using `pass`, just like our first Python class exercise. We have not added a layer or any learned parameters. In this empty example, we have not defined our own `__init__`, so initialization uses the inherited method. We will explicitly call parent initialization when adding our own `__init__` in Lesson 02.

`isinstance(empty_model, torch.nn.Module)` checks whether our object is an instance of that base class or a class inheriting from it.

**Before running:** does inheriting from `Module` create weights by itself? How many scalar parameter values should this empty model report?

**My prediction:** _write here_


In [ ]:
# Our class inherits model features from PyTorch's Module class.
class EmptyModel(torch.nn.Module):
    pass  # No layer or learned parameters have been added yet.

empty_model = EmptyModel()

print("Class:", type(empty_model).__name__)
print("Is a PyTorch module:", isinstance(empty_model, torch.nn.Module))
print("Total learned parameter values:", sum(p.numel() for p in empty_model.parameters()))


**My observation and explanation:** _write here_

<a id="model-class"></a>
## Lesson 02: initialize a layer and define the forward calculation

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

Define `TwoInputModel` with the same class structure we learned:

- `__init__` initializes the new object.
- `super().__init__()` runs `Module` initialization for this object. It sets up the model's bookkeeping and must run before assigning the layer below.
- `self.linear = torch.nn.Linear(2, 1)` creates and stores a layer on this object. The name `linear` is our chosen attribute name. PyTorch registers the layer as part of the model, and its two weights and one bias are discoverable through `model.parameters()`.
- `forward(self, x)` describes how inputs become predictions. It sends `x` through the stored layer.

We used a method named `predict` in plain Python. For a custom PyTorch module, define `forward` and make predictions with `model(x)`. PyTorch's module call uses the forward calculation. We will try that in Lesson 03.

`model.named_parameters()` also supplies names. `linear.weight` means the weight owned by the layer stored as `model.linear`; `linear.bias` is its bias. These are parameter tensors with three scalar values in total.

**Before running:** will storing a layer inside this class add extra weights beyond the layer's two weights and one bias?

**My prediction:** _write here_


In [ ]:
class TwoInputModel(torch.nn.Module):
    def __init__(self):
        super().__init__()  # Initialize PyTorch's model bookkeeping first.
        self.linear = torch.nn.Linear(2, 1)  # Store a two-input, one-output layer.

    def forward(self, x):
        return self.linear(x)  # Send inputs through this object's stored layer.

# Create one instance of our custom model class.
model = TwoInputModel()
print("Model structure:")
print(model)

# PyTorch finds parameters in the layer registered on this model.
for name, parameter in model.named_parameters():
    print("Parameter:", name, "| shape:", parameter.shape)
print("Total learned parameter values:", sum(p.numel() for p in model.parameters()))


**My observation and explanation:** _write here_

<a id="predictions"></a>
## Lesson 03: call the model using known parameter values

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#weighted-sum).

Set both weights to `1` and bias to `0` for a familiar starting point. We now access the layer's values through `model.linear.weight` and `model.linear.bias` because the layer is an attribute of our model object.

`model(x)` calls the module's forward calculation, which calls `self.linear(x)`. The output remains one prediction per example. We evaluate here with `no_grad()` because this cell only inspects predictions. Training will make a fresh forward pass with gradient tracking in the next lesson.

**Before running:** predict the outputs for `[1, 0]` and `[0, 1]` with these parameters.

**My predictions:** _write here_


In [ ]:
# Assign a known starting point to the layer inside our model.
with torch.no_grad():
    model.linear.weight.fill_(1.)
    model.linear.bias.fill_(0.)

x = torch.tensor([[1., 0.], [0., 1.]])
with torch.no_grad():
    predictions = model(x)  # PyTorch uses our forward method.

print("Weights:", model.linear.weight.detach().flatten().tolist())
print("Bias:", model.linear.bias.item())
print("Input shape:", x.shape)
print("Output shape:", predictions.shape)
print("Predictions:", predictions.flatten().tolist())


**My observation and explanation:** _write here_

<a id="training"></a>
## Lesson 04: apply the familiar training step to the whole model

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#feature-gradients).

Reset the parameters so this experiment can be repeated. Use the same two-example batch and targets from Notebook 03: `[1, 0] → 5` and `[0, 1] → 3`.

Pass `model.parameters()` to SGD. It finds the registered layer's existing parameters, so the optimizer updates the weight and bias used by our forward method. The class itself adds no extra trainable values.

The familiar sequence remains: clear gradients, make predictions, measure mean loss, backward, and update. Do the training forward pass outside `no_grad()` so gradients can be calculated. Use a fresh forward pass inside `no_grad()` to evaluate the updated model.

**Before running:** compare this experiment with Notebook 03's two-example batch update. Do you expect different gradients or update values just because we wrapped the layer in our own class?

**My prediction:** _write here_


In [ ]:
# Reset once before this repeatable, single-update experiment.
with torch.no_grad():
    model.linear.weight.fill_(1.)
    model.linear.bias.fill_(0.)

y = torch.tensor([[5.], [3.]])
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0, weight_decay=0)

optimizer.zero_grad(set_to_none=True)  # Clear gradients for the model's parameters.
predictions = model(x)                # Run our model's forward calculation.
loss = loss_fn(predictions, y)
loss.backward()                       # Calculate gradients through the stored layer.
print("Weight gradients:", model.linear.weight.grad.flatten().tolist())
print("Bias gradient:", model.linear.bias.grad.item())
optimizer.step()                      # Update the registered weights and bias.

# Measure predictions and mean loss after the update.
with torch.no_grad():
    updated_predictions = model(x)
    updated_loss = loss_fn(updated_predictions, y)

print("Loss before update:", loss.item())
print("Updated weights:", model.linear.weight.detach().flatten().tolist())
print("Updated bias:", model.linear.bias.item())
print("Updated predictions:", updated_predictions.flatten().tolist())
print("Loss after update:", updated_loss.item())


**My observation and explanation:** _write here_


<a id="model-state"></a>
## Lesson 05: inspect the trained model state

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

`model.state_dict()` returns a dictionary containing named model-state values. A dictionary associates each name (a key) with a value. For our model, those values are the weight and bias tensors of its stored Linear layer.

We have seen their names before: `linear.weight` and `linear.bias`. After Lesson 04, they contain weights approximately `[1.2, 1.1]` and bias approximately `[0.3]`. The dictionary has two entries, but three scalar values: the weight tensor contains two numbers.

In more complex models, a state dictionary also includes registered persistent buffers, such as running statistics. Our current model has no such additional state. It does not include our Python class definition, the optimizer state, or the computed gradients.

For comparison, `model.parameters()` supplies parameter objects to the optimizer; `model.state_dict()` supplies a named mapping of state values for inspection and later saving/loading.

Run this cell after Lesson 04. Inspecting state does not change the parameters or perform training. This call alone does not write a checkpoint file. The returned tensors share data with the model; they are not an independent best-checkpoint snapshot. When we save a best checkpoint in memory later, we will copy the values so further training cannot change that saved snapshot.

**Before running:** should this dictionary contain two entries or three? Which entry contains two scalar values?

**My prediction:** _write here_


In [ ]:
# Inspect named state values from the current trained model.
model_state = model.state_dict()
print("State entry names:", list(model_state.keys()))

# Each dictionary entry has a name and a tensor containing the current values.
for name, value in model_state.items():
    print("Name:", name, "| shape:", value.shape, "| values:", value.flatten().tolist())

print("Dictionary entry count:", len(model_state))
print("Total scalar values:", sum(value.numel() for value in model_state.values()))


**My observation and explanation:** _write here_


<a id="load-state"></a>
## Lesson 06: load the learned state into another model

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

`new_model = TwoInputModel()` creates a separate model object with freshly initialized parameters. It has the same class and layer structure as our trained model, but creating it does not copy the trained values.

`new_model.load_state_dict(model_state)` copies the supplied state values into the matching parameters and buffers of the receiving model. Here, the names `linear.weight` and `linear.bias` and their tensor shapes match because we created the same model class.

First inspect the new model's predictions before loading, which can vary because its initial parameter values are random. Then load the learned state and compare predictions on the same inputs. The original and receiving models remain separate objects; the default loading operation copies the values into the receiving model's own parameters. No training step or backward calculation is performed.

Run this cell after Lesson 05, so `model_state` contains our current trained values. We are loading a state dictionary already in memory. We will study saving and loading a checkpoint file separately. The model's state dictionary does not recreate the Python class definition, so we first construct the compatible model object ourselves.

**Before running:** after loading the state, should both models give the same predictions for the same inputs? Does equal parameter data mean they become the same Python object?

**My predictions and explanation:** _write here_


In [ ]:
# Create a separate instance: same model structure, freshly initialized values.
new_model = TwoInputModel()
with torch.no_grad():
    predictions_before_loading = new_model(x)
print("Fresh weights:", new_model.linear.weight.detach().flatten().tolist())
print("Fresh bias:", new_model.linear.bias.item())
print("Predictions before loading:", predictions_before_loading.flatten().tolist())

# Copy the learned values into this new model's own parameters.
load_result = new_model.load_state_dict(model_state)
print("Load result:", load_result)

# Evaluate both models on identical inputs, without recording for backward.
with torch.no_grad():
    original_predictions = model(x)
    restored_predictions = new_model(x)
    restored_loss = loss_fn(restored_predictions, y)

print("Loaded weights:", new_model.linear.weight.detach().flatten().tolist())
print("Loaded bias:", new_model.linear.bias.item())
print("Original predictions:", original_predictions.flatten().tolist())
print("Restored predictions:", restored_predictions.flatten().tolist())
print("Restored loss:", restored_loss.item())
print("Predictions agree:", torch.allclose(original_predictions, restored_predictions))
print("Are they the same object?", new_model is model)


**My observation and explanation:** _write here_


<a id="save-file"></a>
## Lesson 07: save the current state to a checkpoint file

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

The learned values we need to preserve are the weights and bias. Our previous state dictionary lived in Python's memory. `torch.save(model.state_dict(), checkpoint_path)` writes those current state values to a file, so we can read them in a later Python session.

Keep the Python class definition in the notebook. The state-only checkpoint contains named tensors, not the class definition, training data, or optimizer state. We already know how to create a compatible model and use `load_state_dict()`; the next lesson will first read the dictionary from this file.

`Path` from Python's `pathlib` represents a filesystem path. We choose a `checkpoints` folder alongside this notebook in our course directory. `mkdir(parents=True, exist_ok=True)` creates the folder if needed. The filename is `two_input_model_state.pt`; `.pt` is a conventional PyTorch filename extension. If you move the course folder, update this path.

Run this cell after Lesson 06. It saves the current original model state after our one-update experiment, whose values are approximately weights `[1.2, 1.1]` and bias `0.3`. Saving does not perform another training update. Rerunning the cell replaces this example checkpoint with the current state at that time.

The file records values at the moment they are saved; further parameter updates do not change the file unless we explicitly save again. Selecting which training checkpoint to save is a separate decision, guided by validation loss as we learned earlier.

**Before running:** will saving need a backward call or change the weights and bias?

**My prediction:** _write here_


In [ ]:
from pathlib import Path

# Keep this example checkpoint alongside the notebook in our course folder.
checkpoint_dir = course_checkpoint_dir()
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / "two_input_model_state.pt"

# Serialize the current named tensors to disk; this does not train the model.
state_to_save = model.state_dict()
torch.save(state_to_save, checkpoint_path)

print("Saved checkpoint:", checkpoint_path)
print("File exists:", checkpoint_path.exists())
print("Saved entry names:", list(state_to_save.keys()))
print("Saved weights:", state_to_save["linear.weight"].flatten().tolist())
print("Saved bias:", state_to_save["linear.bias"].flatten().tolist())


**My observation and explanation:** _write here_


<a id="restore-file"></a>
## Lesson 08: restore the model from its checkpoint file

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#model-state).

The file created in Lesson 07 persists after the Python session ends. To restore a model, use three separate actions:

1. `torch.load(checkpoint_path, ...)` reads the saved state dictionary from the file.
2. `TwoInputModel()` constructs a new model object with a compatible structure.
3. `model_from_file.load_state_dict(file_state)` copies the read values into that object's parameters.

`torch.load()` reads the data we saved; it does not construct our Python model class from this state-only checkpoint. `load_state_dict()` applies those data to the model we constructed.

The code uses `map_location="cpu"` to read the tensors onto the CPU and `weights_only=True` to restrict loading to tensors and supported simple data. We are reading our own state-only example checkpoint.

For prediction, set evaluation mode with `eval()` and use `no_grad()` to skip recording. As reviewed earlier, these have different jobs; evaluation mode alone does not disable gradient recording. Our model contains only a Linear layer, whose prediction calculation is the same in either mode.

Run this after Lesson 07 has created the file. This cell needs only imported PyTorch, the `TwoInputModel` class definition, and that file. It defines its own input, targets, and loss function and does not use the original trained model, in-memory state dictionary, or optimizer.

If you want to try restoration after restarting the kernel, run Setup and Lesson 02 to recreate the class definition, then run this cell directly. You do not need to run the training or saving cells again when the file already exists.

**Before running:** will reading the file itself create a model instance? Will this restored model need training again to reproduce the saved predictions?

**My predictions and explanation:** _write here_


In [ ]:
from pathlib import Path

# Read the checkpoint created in Lesson 07, independently of old Python variables.
checkpoint_path = (course_checkpoint_dir() / "two_input_model_state.pt")
file_state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

# Construct the model structure, then apply the saved values to its parameters.
model_from_file = TwoInputModel()
load_result = model_from_file.load_state_dict(file_state)
model_from_file.eval()  # Select evaluation mode; gradient recording is controlled separately.

# Supply our familiar examples and targets afresh.
restore_x = torch.tensor([[1., 0.], [0., 1.]])
restore_y = torch.tensor([[5.], [3.]])
restore_loss_fn = torch.nn.MSELoss()
with torch.no_grad():
    predictions_from_file = model_from_file(restore_x)
    loss_from_file = restore_loss_fn(predictions_from_file, restore_y)

print("Read checkpoint:", checkpoint_path)
print("Load result:", load_result)
print("Restored weights:", model_from_file.linear.weight.detach().flatten().tolist())
print("Restored bias:", model_from_file.linear.bias.item())
print("Predictions from file:", predictions_from_file.flatten().tolist())
print("Loss from file:", loss_from_file.item())
# Compare with the predictions we recorded after the saved one-update experiment.
expected_predictions = torch.tensor([[1.5], [1.4]])
print("Matches recorded predictions:", torch.allclose(predictions_from_file, expected_predictions))


**My observation and explanation:** _write here_


<a id="optimizer-state"></a>
## Lesson 09: inspect optimizer memory for resuming training

**Mathematical view:** [Related explanation and worked example](../04_optimizers_and_momentum/00_math_intuition.ipynb#state).

Restoring the learned weights and bias reproduces predictions from the saved model. To resume the same momentum training trajectory, we also need the optimizer's stored history and settings.

The earlier plain SGD experiment used momentum 0. For this new experiment, create a separate model with weights `[1, 1]`, bias `0`, and SGD with learning rate `0.05` and momentum `0.9`. Use the same two examples and targets. One update creates momentum buffers for the weight and bias parameters.

As we learned in Chapter 04, the first momentum buffer is the first gradient. This first update therefore matches plain SGD. On the next update, the direction combines `0.9 × previous buffer + fresh gradient`.

`momentum_model.state_dict()` contains its model values. The momentum buffers belong to `momentum_optimizer`, so they are found in `momentum_optimizer.state_dict()` instead. That optimizer dictionary has two main entries:

- `state`: optimizer memory associated with its parameters, including the momentum buffers here. It does not contain the learned parameter values themselves.
- `param_groups`: settings such as learning rate and momentum, together with the identifiers of the parameters those settings apply to.

The loop below uses parameter names from the model to label the corresponding optimizer buffers. The optimizer tracks a buffer for the weight tensor and a buffer for the bias tensor; the weight buffer has two entries, one for each weight.

This cell creates its own experiment and leaves the earlier model and checkpoint file alone. Run after Setup and Lesson 02 have defined the class. The next lesson will save the model and optimizer state together.

**Before running:** after one update, what buffer values do you expect for weight 1, weight 2, and bias? Does the model state dictionary itself contain those buffers?

**My prediction:** _write here_


In [ ]:
# Create a separate, repeatable momentum experiment.
momentum_model = TwoInputModel()
with torch.no_grad():
    momentum_model.linear.weight.fill_(1.)
    momentum_model.linear.bias.fill_(0.)

resume_x = torch.tensor([[1., 0.], [0., 1.]])
resume_y = torch.tensor([[5.], [3.]])
resume_loss_fn = torch.nn.MSELoss()
momentum_optimizer = torch.optim.SGD(momentum_model.parameters(), lr=0.05, momentum=0.9, weight_decay=0)
print("Optimizer memory before first step:", momentum_optimizer.state_dict()["state"])

# One tracked training step creates the optimizer's momentum buffers.
momentum_optimizer.zero_grad(set_to_none=True)
resume_loss = resume_loss_fn(momentum_model(resume_x), resume_y)
resume_loss.backward()
momentum_optimizer.step()
updates_completed = 1

print("Weights after update:", momentum_model.linear.weight.detach().flatten().tolist())
print("Bias after update:", momentum_model.linear.bias.item())

# Label each stored buffer using the name of its associated model parameter.
for name, parameter in momentum_model.named_parameters():
    buffer = momentum_optimizer.state[parameter]["momentum_buffer"]
    print("Momentum buffer for", name, ":", buffer.flatten().tolist())

optimizer_state = momentum_optimizer.state_dict()
print("Model state entry names:", list(momentum_model.state_dict().keys()))
print("Optimizer state entry names:", list(optimizer_state.keys()))
print("Stored learning rate:", optimizer_state["param_groups"][0]["lr"])
print("Stored momentum:", optimizer_state["param_groups"][0]["momentum"])


**My observation and explanation:** _write here_


<a id="training-checkpoint"></a>
## Lesson 10: save a checkpoint for the momentum experiment

**Mathematical view:** [Related explanation and worked example](../04_optimizers_and_momentum/00_math_intuition.ipynb#state).

Our momentum experiment has completed one update. To preserve the state needed to resume this experiment, put three items into a dictionary and save it:

| Checkpoint key | What it preserves |
|---|---|
| `model_state` | Current weights and bias |
| `optimizer_state` | Momentum buffers and optimizer settings |
| `updates_completed` | Our record of how many updates have already run |

The keys are names we choose for this checkpoint format. `model_state` and `optimizer_state` each contain their own nested state dictionaries. `updates_completed` is an ordinary integer. The counter helps us label our progress; the optimizer uses its own saved buffers and settings to calculate the next update.

Run this cell immediately after Lesson 09, before performing another momentum update. It saves the model and optimizer at the same point in training. Both are needed to continue our momentum run using its existing history. Our fixed-batch, deterministic example has no additional scheduler or changing data-order state.

Use the new filename `two_input_momentum_training.pt` in the chapter's checkpoint folder, keeping the earlier model-state file available for Lesson 08. Rerunning this cell replaces this example training checkpoint with the current momentum experiment state.

The dictionary assembled here contains references to the current tensors; we immediately serialize it to disk. The saved file records their values at save time. Further training does not change the file unless we save again. If we instead kept a best-checkpoint snapshot only in memory while training continued, we would need independent copies of the state values.

Saving does not train the model or run backward. The next lesson will read this file and restore both the model and optimizer.

**Before running:** which dictionary item stores momentum memory? Why also save a count of completed updates?

**My answer:** _write here_


In [ ]:
from pathlib import Path

# Capture the model and optimizer at the same point in the momentum experiment.
training_checkpoint = {
    "model_state": momentum_model.state_dict(),
    "optimizer_state": momentum_optimizer.state_dict(),
    "updates_completed": updates_completed,
}

# Keep this checkpoint separate from the earlier model-state example file.
checkpoint_dir = course_checkpoint_dir()
checkpoint_dir.mkdir(parents=True, exist_ok=True)
training_checkpoint_path = checkpoint_dir / "two_input_momentum_training.pt"
torch.save(training_checkpoint, training_checkpoint_path)

print("Saved training checkpoint:", training_checkpoint_path)
print("File exists:", training_checkpoint_path.exists())
print("Checkpoint keys:", list(training_checkpoint.keys()))
print("Completed updates:", training_checkpoint["updates_completed"])
print("Model state entries:", list(training_checkpoint["model_state"].keys()))
print("Optimizer state entries:", list(training_checkpoint["optimizer_state"].keys()))
print("Saved learning rate:", training_checkpoint["optimizer_state"]["param_groups"][0]["lr"])
print("Saved momentum:", training_checkpoint["optimizer_state"]["param_groups"][0]["momentum"])


**My observation and explanation:** _write here_


<a id="restore-training"></a>
## Lesson 11: restore the model and optimizer together

**Mathematical view:** [Related explanation and worked example](../04_optimizers_and_momentum/00_math_intuition.ipynb#state).

Read the training checkpoint from Lesson 10, then restore each part to the object it belongs to:

1. Create a compatible `TwoInputModel` and load `checkpoint["model_state"]` into it.
2. Create SGD using **that restored model's parameters**.
3. Load `checkpoint["optimizer_state"]` into that optimizer.
4. Recover `checkpoint["updates_completed"]` to record where training stopped.

The outer checkpoint dictionary has three entries. Each `load_state_dict()` call receives its own nested state dictionary: the model gets the model values, and the optimizer gets optimizer history and settings. Use the same model structure and optimizer parameter-group arrangement as the saved experiment.

`resumed_model.train()` selects training mode; it does not perform any training update. This cell inspects the loaded values, momentum buffers, and settings, then evaluates current predictions with no-grad. The completed-update count should remain `1`, because we have not run the next update yet.

The examples and targets are supplied again in this cell. They were not stored in our checkpoint. The next lesson will calculate fresh gradients at the restored parameter values and combine them with the restored momentum memory.

Run after Lesson 10 has written its file. In a fresh kernel, you only need Setup and Lesson 02 to define the class before running this cell; the saved file supplies the training state. No original model or optimizer instance is needed.

**Before running:** should the restored optimizer's buffers be empty? Does constructing the new optimizer alone recreate the previous momentum history?

**My predictions and explanation:** _write here_


In [ ]:
from pathlib import Path

# Read all three saved pieces from the training checkpoint.
training_checkpoint_path = (course_checkpoint_dir() / "two_input_momentum_training.pt")
loaded_training_checkpoint = torch.load(training_checkpoint_path, map_location="cpu", weights_only=True)

# Restore model values first, using the same class and layer structure.
resumed_model = TwoInputModel()
model_load_result = resumed_model.load_state_dict(loaded_training_checkpoint["model_state"])

# Register this new model's own parameters with a matching SGD optimizer.
resumed_optimizer = torch.optim.SGD(resumed_model.parameters(), lr=0.05, momentum=0.9, weight_decay=0)
resumed_optimizer.load_state_dict(loaded_training_checkpoint["optimizer_state"])
resumed_updates_completed = loaded_training_checkpoint["updates_completed"]
resumed_model.train()  # Select training mode; this does not run a training step.

# Recreate the batch explicitly; the checkpoint stores state, not our dataset.
resumed_x = torch.tensor([[1., 0.], [0., 1.]])
resumed_y = torch.tensor([[5.], [3.]])
resumed_loss_fn = torch.nn.MSELoss()
with torch.no_grad():
    predictions_before_resuming = resumed_model(resumed_x)
    loss_before_resuming = resumed_loss_fn(predictions_before_resuming, resumed_y)

print("Model load result:", model_load_result)
print("Completed updates restored:", resumed_updates_completed)
print("Restored weights:", resumed_model.linear.weight.detach().flatten().tolist())
print("Restored bias:", resumed_model.linear.bias.item())
for name, parameter in resumed_model.named_parameters():
    buffer = resumed_optimizer.state[parameter]["momentum_buffer"]
    print("Restored momentum buffer for", name, ":", buffer.flatten().tolist())
print("Restored learning rate:", resumed_optimizer.param_groups[0]["lr"])
print("Restored momentum:", resumed_optimizer.param_groups[0]["momentum"])
print("Predictions before next update:", predictions_before_resuming.flatten().tolist())
print("Loss before next update:", loss_before_resuming.item())


**My observation and explanation:** _write here_


<a id="resume-update"></a>
## Lesson 12: continue with the next momentum update

**Mathematical view:** [Related explanation and worked example](../04_optimizers_and_momentum/00_math_intuition.ipynb#momentum).

Lesson 11 restored the state after one update: weights approximately `[1.2, 1.1]`, bias `0.3`, and the optimizer's stored momentum buffers. Now perform update two.

Calculate fresh gradients at those current parameter values. The saved momentum buffers are history, not substitutes for fresh gradients. `zero_grad()` clears the parameter gradient fields and leaves the optimizer's momentum buffers intact.

For our SGD settings, the next direction is `0.9 × previous buffer + fresh gradient`. `optimizer.step()` calculates that direction, stores it as the new buffer, and subtracts `learning rate × direction` from the parameter. The printed buffer after the step therefore shows the direction used for this update.

Run this cell **once after Lesson 11**. To repeat the same restored-update experiment, rerun Lesson 11 first to reload the checkpoint. Repeating this cell alone continues to further updates using the evolving model and optimizer state.

The progress counter increments after the update. We evaluate a fresh loss after updating inside `no_grad()`. This is training loss on the same two-example batch.

The checkpoint file still records the save point after update one. To save newer progress, we would assemble a new checkpoint from the current resumed model state, optimizer state, and completed-update counter and explicitly save it. This cell continues training in memory.

**Before running:** should we reuse the old gradients or calculate fresh ones? Will clearing gradients erase the restored momentum buffers?

**My prediction and explanation:** _write here_


In [ ]:
# Inspect the stored history before calculating the next update.
print("Completed updates before step:", resumed_updates_completed)
for name, parameter in resumed_model.named_parameters():
    buffer = resumed_optimizer.state[parameter]["momentum_buffer"]
    print("Previous momentum buffer for", name, ":", buffer.flatten().tolist())

# Fresh gradients come from the current restored parameters and current batch.
resumed_optimizer.zero_grad(set_to_none=True)
next_predictions = resumed_model(resumed_x)
next_loss = resumed_loss_fn(next_predictions, resumed_y)
next_loss.backward()
print("Fresh weight gradients:", resumed_model.linear.weight.grad.flatten().tolist())
print("Fresh bias gradient:", resumed_model.linear.bias.grad.item())

# SGD combines these gradients with its restored momentum memory.
resumed_optimizer.step()
resumed_updates_completed += 1
for name, parameter in resumed_model.named_parameters():
    direction = resumed_optimizer.state[parameter]["momentum_buffer"]
    print("Direction used for", name, ":", direction.flatten().tolist())

# Recalculate predictions and loss after the resumed update.
with torch.no_grad():
    predictions_after_resuming = resumed_model(resumed_x)
    loss_after_resuming = resumed_loss_fn(predictions_after_resuming, resumed_y)

print("Completed updates after step:", resumed_updates_completed)
print("Updated weights:", resumed_model.linear.weight.detach().flatten().tolist())
print("Updated bias:", resumed_model.linear.bias.item())
print("Updated predictions:", predictions_after_resuming.flatten().tolist())
print("Loss before this update:", next_loss.item())
print("Loss after this update:", loss_after_resuming.item())


**My observation and explanation:** _write here_


<a id="resume-comparison"></a>
## Lesson 13: compare three ways to reach the next update

**Mathematical view:** [Related explanation and worked example](../04_optimizers_and_momentum/00_math_intuition.ipynb#state).

Setting `momentum=0.9` chooses the rule's coefficient. It does not reconstruct previous gradient history. The buffer is in the saved optimizer state.

Compare three cases, all using the same batch, targets, learning rate 0.05, and momentum 0.9:

| Case | Starting point | Updates to run here |
|---|---|---:|
| Uninterrupted | Weights `[1, 1]`, bias 0, fresh optimizer | Two |
| Model + optimizer restored | Both states saved after update one | One more |
| Model only restored | Model state saved after update one, fresh optimizer | One more |

All three results represent two total updates of the model. The latter two models start with identical parameters, but only the full restoration has the saved momentum buffers.

At the saved point, the fresh weight 1 gradient is -3.5. The restored previous buffer is -4, so full restoration uses direction `0.9 × (-4) + (-3.5) = -7.1`. A new optimizer with no history uses its first fresh gradient, -3.5, as the initial buffer. It still has momentum configured as 0.9.

This exercise reads the checkpoint saved after one update and constructs fresh instances for all three cases, so it can be rerun independently of Lesson 12's current objects. It does not rewrite the file. In a fresh kernel, run Setup and Lesson 02 to define the class, then this cell once the checkpoint exists.

The code's `cases` list holds a label, model, optimizer, and number of additional updates for each experiment. The nested loop performs the same training sequence for each case.

**Before running:** which restoration should match uninterrupted training? Will the fresh gradient be different between the two restored models on their first additional update, or will their update direction differ because of optimizer history?

**My prediction and explanation:** _write here_


In [ ]:
from pathlib import Path

# Start from the saved point after one momentum update.
comparison_path = (course_checkpoint_dir() / "two_input_momentum_training.pt")
comparison_checkpoint = torch.load(comparison_path, map_location="cpu", weights_only=True)
comparison_x = torch.tensor([[1., 0.], [0., 1.]])
comparison_y = torch.tensor([[5.], [3.]])
comparison_loss_fn = torch.nn.MSELoss()

# Case 1: run from the known original parameters without a pause.
reference_model = TwoInputModel()
with torch.no_grad():
    reference_model.linear.weight.fill_(1.)
    reference_model.linear.bias.fill_(0.)
reference_optimizer = torch.optim.SGD(reference_model.parameters(), lr=0.05, momentum=0.9, weight_decay=0)

# Case 2: restore both parameter values and optimizer history.
full_model = TwoInputModel()
full_model.load_state_dict(comparison_checkpoint["model_state"])
full_optimizer = torch.optim.SGD(full_model.parameters(), lr=0.05, momentum=0.9, weight_decay=0)
full_optimizer.load_state_dict(comparison_checkpoint["optimizer_state"])

# Case 3: restore the same parameter values, but leave optimizer memory fresh.
model_only_model = TwoInputModel()
model_only_model.load_state_dict(comparison_checkpoint["model_state"])
model_only_optimizer = torch.optim.SGD(model_only_model.parameters(), lr=0.05, momentum=0.9, weight_decay=0)

# Each tuple contains a label, model, optimizer, and additional-update count.
total_updates = comparison_checkpoint["updates_completed"] + 1
cases = [
    ("Uninterrupted", reference_model, reference_optimizer, total_updates),
    ("Model + optimizer restored", full_model, full_optimizer, 1),
    ("Model only restored", model_only_model, model_only_optimizer, 1),
]
print("Compare after total model updates:", total_updates)
print("Case | grad_w1 | direction_w1 | weight1 | weight2 | bias | training loss")

for label, case_model, case_optimizer, additional_updates in cases:
    case_model.train()
    for _ in range(additional_updates):
        case_optimizer.zero_grad(set_to_none=True)
        case_loss = comparison_loss_fn(case_model(comparison_x), comparison_y)
        case_loss.backward()
        case_optimizer.step()

    # Inspect the last gradient and the momentum direction used for the last update.
    gradient_w1 = case_model.linear.weight.grad[0, 0].item()
    direction_w1 = case_optimizer.state[case_model.linear.weight]["momentum_buffer"][0, 0].item()
    with torch.no_grad():
        final_case_loss = comparison_loss_fn(case_model(comparison_x), comparison_y).item()
    print(
        f"{label:26s} | {gradient_w1:7.4f} | {direction_w1:12.4f} | "
        f"{case_model.linear.weight[0, 0].item():.4f} | "
        f"{case_model.linear.weight[0, 1].item():.4f} | "
        f"{case_model.linear.bias.item():.4f} | {final_case_loss:.6f}"
    )

# Compare all learned parameter values, rather than only one prediction.
full_matches = all(torch.allclose(a, b) for a, b in zip(reference_model.parameters(), full_model.parameters()))
model_only_matches = all(torch.allclose(a, b) for a, b in zip(reference_model.parameters(), model_only_model.parameters()))
print("Full restoration matches uninterrupted:", full_matches)
print("Model-only restoration matches uninterrupted:", model_only_matches)


**My observation and explanation:** _write here_

Both restored cases initially have the same model values, so the same batch gives them the same fresh gradients. Their optimizer histories differ, so their next update directions differ. This fixed-data experiment checks exact continuation; it does not establish that one momentum setting or training path is always best.

## Vocabulary to revisit

| Term | Meaning in this model |
|---|---|
| Inheritance | `TwoInputModel` uses features supplied by `Module` |
| Parent / base class | `torch.nn.Module` |
| `super().__init__()` | Run the parent's initialization for this instance |
| `self.linear` | The layer stored on this particular model object |
| `forward` | The calculation used to turn inputs into outputs |
| `model.parameters()` | Discover parameters registered in this model and its layers |
| `model.state_dict()` | Inspect named model-state values for later saving and loading |
| `new_model.load_state_dict(state)` | Copy compatible state values into another model instance |
| `torch.save(state, path)` | Write state values to a checkpoint file |
| `torch.load(path, ...)` | Read saved state values from a checkpoint file |
| `optimizer.state_dict()` | Collect optimizer history and settings for later restoration |
| `optimizer.load_state_dict(state)` | Restore saved optimizer history and settings |

## Documentation

- [PyTorch: Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)
- [PyTorch: Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)
- [PyTorch: SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)

- [PyTorch: torch.save](https://docs.pytorch.org/docs/stable/generated/torch.save.html)

- [PyTorch: torch.load](https://docs.pytorch.org/docs/stable/generated/torch.load.html)

- [PyTorch: Optimizer.state_dict](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.state_dict.html)

- [PyTorch: Optimizer.load_state_dict](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.load_state_dict.html)
